# Ishara Backend on Colab — Gemma 4 via Ollama

Run this notebook on a Colab GPU runtime to host the Ishara FastAPI backend
with `gemma4:latest` and expose it to the Flutter app over a public Cloudflare
tunnel.

**Setup**
1. Runtime → Change runtime type → **T4 GPU** (or any GPU runtime).
2. Run cells top to bottom.
3. The tunnel cell prints a `https://*.trycloudflare.com` URL.
4. In the Ishara app, open Settings and set the host to that URL (drop the
   `https://` prefix and use port `443`).

Keep this notebook running for the duration of the demo.


## 1. Install Ollama


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


## 2. Start the Ollama server


In [ ]:
import os, subprocess, time, requests

os.makedirs('/content/ishara_logs', exist_ok=True)
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('/content/ishara_logs/ollama.log', 'w'),
    stderr=subprocess.STDOUT,
)
for _ in range(30):
    try:
        if requests.get('http://localhost:11434/api/tags', timeout=1).ok:
            print('Ollama is up.')
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Ollama did not start — check /content/ishara_logs/ollama.log')


## 3. Pull `gemma4:latest`

This downloads ~10 GB. With a T4 GPU runtime the model fits in VRAM and runs
fast — exactly what we couldn't get on the local Mac Air.


In [ ]:
!ollama pull gemma4:latest
!ollama list


## 4. Clone the Ishara repo and install backend deps


In [ ]:
!if [ -d /content/ishara ]; then cd /content/ishara && git pull; else git clone https://github.com/Medialordofficial/Ishara.git /content/ishara; fi
!pip install -q -r /content/ishara/backend/requirements.txt


## 5. Configure the backend for Gemma 4


In [ ]:
env_text = """ISHARA_API_KEY=
ISHARA_RATE_LIMIT=120
OLLAMA_URL=http://localhost:11434
ISHARA_MODEL=gemma4:latest
ISHARA_FAST_MODEL=gemma4:latest
ISHARA_FULL_MODEL=gemma4:latest
ISHARA_CORS_ORIGINS=*
ISHARA_OLLAMA_TIMEOUT=300
LOG_LEVEL=INFO
"""
open('/content/ishara/backend/.env', 'w').write(env_text)
print(env_text)


## 6. Launch the FastAPI backend


In [ ]:
import os, subprocess, time, requests

backend_env = os.environ.copy()
for line in open('/content/ishara/backend/.env'):
    line = line.strip()
    if line and not line.startswith('#') and '=' in line:
        k, v = line.split('=', 1)
        backend_env[k] = v

backend_proc = subprocess.Popen(
    ['uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/ishara/backend',
    env=backend_env,
    stdout=open('/content/ishara_logs/backend.log', 'w'),
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.ok:
            print('Backend up:', r.json())
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('--- backend log ---')
    print(open('/content/ishara_logs/backend.log').read())
    raise RuntimeError('Backend did not start')


## 7. Open a public Cloudflare tunnel

This prints a `https://<random>.trycloudflare.com` URL. Use it as the backend
host in the Ishara app's Settings screen (drop the scheme, port `443`).


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess, re, threading

public_url = None
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

def _stream():
    global public_url
    pattern = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
    for line in tunnel.stdout:
        print(line, end='')
        if public_url is None:
            m = pattern.search(line)
            if m:
                public_url = m.group(0)
                print('\n' + '=' * 60)
                print('PUBLIC BACKEND URL:', public_url)
                print('In the Ishara app Settings screen, set:')
                print('  Host:', public_url.replace('https://', ''))
                print('  Port: 443  (and enable HTTPS)')
                print('=' * 60 + '\n', flush=True)

threading.Thread(target=_stream, daemon=True).start()

import time
for _ in range(30):
    if public_url:
        break
    time.sleep(1)
print('\nReady. public_url =', public_url)


## 8. Smoke test the public endpoint


In [ ]:
import requests, time

assert public_url, 'Tunnel did not produce a URL — re-run the previous cell.'
print('Health:', requests.get(public_url + '/health', timeout=15).json())

t0 = time.time()
r = requests.post(
    public_url + '/chat',
    json={'message': 'smoke kitchen alone scared what do'},
    timeout=300,
)
print(f'/chat took {time.time() - t0:.1f}s')
print(r.json())


## 9. Keep this notebook running

The tunnel and backend stay alive as long as this Colab session is running.
If Colab disconnects, re-run the notebook from the top to get a fresh public
URL.

To inspect logs:
```python
!tail -50 /content/ishara_logs/backend.log
!tail -50 /content/ishara_logs/ollama.log
```
